In [1]:
%load_ext autoreload
%autoreload 2

In [12]:
from pathlib import Path

import numpy as np
import rasterio
import matplotlib.pyplot as plt

import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.strategies import DDPStrategy
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

from dataset.dataset_meta_concat import SegmentationDatasetMetaConcat
from dataset.datamodule_meta_concat import DeepLabV3PlusMetaConcat
from utils.albumentation import augment, transform
from utils.plotting import plot_image_mask

In [3]:
eval_root = Path("C:/Users/Administrator/PythonProjects/landcover_classification/FLAIR1/results")
data_root = Path("C:/Users/Administrator/PythonProjects/landcover_classification/ML_datasets/FLAIR1/flair_1_toy_dataset")
subfolder = "baseline"

In [13]:
model = DeepLabV3PlusMetaConcat(
    data_dir=data_root,
    num_classes=19, 
    in_channels=5, 
    meta_dim=25, 
    augment=augment,
    transform=transform,
    batch_size=2,
    num_workers=2,
    ignore_index=0,
)

In [84]:
# import json
# json_path = data_root / "flair_1_metadata_aerial"

# json_path = [p for p in Path(json_path).rglob("*.json") if "_geohash" in str(p)][0]

# with open(json_path) as f:
#     metadata = json.load(f)

# image_stem = metadata["IMG_000001"]
# image_stem

In [14]:
model.setup(stage="fit")
train_dataloader = model.train_dataloader()
val_dataloader = model.val_dataloader()

model.setup(stage="test")
test_dataloader = model.test_dataloader()

Lenght full dataset: 200
Lenght training dataset: 160
Lenght validation dataset: 40
Lenght test dataset: 50


In [ ]:
from itertools import islice
num_batches = 1

for dict in islice(train_dataloader, num_batches):
    for image, mask in zip(dict["image"], dict["mask"]):
        print(image)
        print(mask)

In [ ]:
plot_image_mask(data_loader=train_dataloader, take=2, normalized=True)

In [29]:
class_weights = torch.tensor([
    0.0000e+00, 2.8986e-03, 4.7309e-03, 1.8890e-03, 7.4763e-03, 6.0552e-03,
    7.3642e-03, 2.2929e-03, 4.6302e-03, 2.3903e-02, 1.7715e-03, 3.3325e-03,
    7.8365e-03, 6.1710e-01, 6.9375e-01, 4.3067e-01, 0.0000e+00, 8.8827e+00,
    6.3016e+00
])

In [30]:
# Checkpoints
checkpoint_dir = eval_root / subfolder / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True, parents=True)

# experiment = ""
# experiment_dir = root / experiment / "checkpoints"
# checkpoint_path = experiment_dir / ""

# model = SegmentationModule.load_from_checkpoint(
#     checkpoint_path=checkpoint_path,
#     weights_only=False,
#     data_dir=data_root
#     num_classes=19,
#     in_channels=5,
#     ignore_index=19,
#     class_weights=class_weights,
#     learning_rate=1e-03,
#     batch_size=16,
#     num_workers=4
# )

In [31]:
# Callbacks
logger_dir = eval_root / subfolder / "logger"
logger_dir.mkdir(parents=True, exist_ok=True)

csv_logger = CSVLogger(
    save_dir=logger_dir,
    name="csv_logs"
)

tb_logger = TensorBoardLogger(
    logger_dir, 
    name="tb_logs"
)

checkpoint = ModelCheckpoint(
    monitor="val_loss",
    dirpath=checkpoint_dir,
    filename="ckpt-{epoch:02d}-{val_loss:.3f}-{val_iou:.3f}",
    save_top_k=2,
    mode="min",
    save_weights_only=True
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    min_delta=0.00,
    patience=20,
    mode="max",
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

callbacks = [
    checkpoint,
    early_stopping,
    lr_monitor
]


# Define trainer
trainer = Trainer(
    # accelerator="gpu",
    accelerator="cpu",
    max_epochs=200,
    # devices=[1],
    devices="auto",
    # strategy=DDPStrategy(find_unused_parameters=True),
    callbacks=callbacks,
    enable_progress_bar=True,
    logger=[csv_logger, tb_logger],
    log_every_n_steps=10
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
# Train model
trainer.fit(model)


  | Name      | Type                   | Params | Mode  | FLOPs
---------------------------------------------------------------------
0 | model     | DeepLabV3Plus          | 26.7 M | train | 0    
1 | criterion | FocalDiceLoss          | 0      | train | 0    
2 | train_iou | MulticlassJaccardIndex | 0      | train | 0    
3 | val_iou   | MulticlassJaccardIndex | 0      | train | 0    
4 | test_iou  | MulticlassJaccardIndex | 0      | train | 0    
---------------------------------------------------------------------
26.7 M    Trainable params
0         Non-trainable params
26.7 M    Total params
106.754   Total estimated model params size (MB)
213       Modules in train mode
0         Modules in eval mode
0         Total Flops


Lenght full dataset: 200
Lenght training dataset: 160
Lenght validation dataset: 40


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\ProgramData\Miniforge3\envs\bsky_alb_mon\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
